In [2]:
import pandas as pd
import numpy as np
from datasets import load_dataset
import tiktoken
import json
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
import re
from groq import Groq
import os
from dotenv import load_dotenv
import random

load_dotenv()  # Load environment variables from a .env file if present
client = Groq(api_key=os.getenv("groq_api_key"))

d:\Work\Github\google-tunix-kaggle\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Initialize tokenizer for token counting
encoding = tiktoken.get_encoding("cl100k_base")  # GPT-4 tokenizer

def count_tokens(text):
    """Count tokens in text using tiktoken"""
    if text is None:
        return 0
    return len(encoding.encode(str(text)))


def convert_to_pandas(dataset, batch_size=1000):
    """
    Convert a Hugging Face dataset (streaming or not) to a pandas DataFrame in batches.
    """
    df_list = []
    
    i=0
    # Hugging Face streaming datasets use iter() for batching
    for batch in dataset.iter(batch_size=batch_size):
        print(f"Processing batch {i} \n")
        i += 1
        # batch is a dict, convert directly to DataFrame
        df_batch = pd.DataFrame(batch)
        df_list.append(df_batch)
    
    df = pd.concat(df_list, ignore_index=True)
    return df

def sample_hf_data(data, sample_size=1000):

    # Shuffle the dataset (buffer_size controls memory usage)
    shuffled = data.shuffle(buffer_size=sample_size)
    # Take random samples
    sampled_dataset = shuffled.take(sample_size)
    
    return sampled_dataset

def extract_gsm8k_answer(text):
    m = re.search(r"####\s*([0-9]+)", text)
    return m.group(1) if m else None

### Load Different Datasets 

    - We will collect different datasets from many sources and collate them with their category
    - Then we will run the questions through oss120B with low/medum reasoning and get the reasoning and response for them
    - Use it for distillation/GRPO later for code domain


Why not use existing datasets for distillation?

    1. `baseten-admin/gpt-oss120b-generated-perfectblend`
    2. `baseten-admin/gpt-oss120b-generated-magpie-1m-v0.1`
    3. `Jackrong/gpt-oss-120B-distilled-reasoning`

Becuause these datasets although great, were not all created with the Tunix hackathon compute limitation in mind. A lot of them go over 2048 tokens in reasponse and reasoning.
Also, these datsets doesn't cover all domain and we have to do separate classification to idenitfy domain of samples used. We need a curated dataset with perfect blend (pun intented).
So we will create our own dataset questions, ground truth if present and then add reasoning and response from gpt oss 120B







## 1️ Mathematics / Reasoning

Even outside code, math reasoning improves logical step-by-step thinking. Good for both reasoning and accuracy.

| Dataset               | Purpose                              | HuggingFace Link                                                       |
| --------------------- | ------------------------------------ | ---------------------------------------------------------------------- |
| GSM8K                 | Math word problems, reasoning steps  | [GSM8K](https://huggingface.co/datasets/gsm8k)                         |
| MATH (short versions) | Middle/high school problems          | [MATH](https://huggingface.co/datasets/MathematicsDataset/mathematics) |
| SVAMP                 | Small multi-step arithmetic problems | [SVAMP](https://huggingface.co/datasets/svamp)                         |


---

## 2️ Reading Comprehension

Good for general knowledge and context understanding. Helps model answer questions accurately.

| Dataset                | Purpose                                        | HuggingFace Link                                        |
| ---------------------- | ---------------------------------------------- | ------------------------------------------------------- |
| SQuAD v2               | Reading comprehension + unanswerable questions | [SQuAD v2](https://huggingface.co/datasets/squad_v2)    |
| Natural Questions (NQ) | Real-world questions on Wikipedia              | [NQ](https://huggingface.co/datasets/natural_questions) |
| RACE                   | Multiple-choice reading comprehension          | [RACE](https://huggingface.co/datasets/race)            |
| TriviaQA               | Open-domain QA with passages                   | [TriviaQA](https://huggingface.co/datasets/trivia_qa)   |


---

## 3️ Commonsense / Multi-hop Reasoning

To improve answer accuracy, the model should handle implicit knowledge and multi-step reasoning.

| Dataset       | Purpose                       | HuggingFace Link                                                |
| ------------- | ----------------------------- | --------------------------------------------------------------- |
| CommonsenseQA | Commonsense QA with reasoning | [CommonsenseQA](https://huggingface.co/datasets/commonsense_qa) |
| StrategyQA    | Multi-hop reasoning           | [StrategyQA](https://huggingface.co/datasets/strategy_qa)       |
| OpenBookQA    | Scientific commonsense        | [OpenBookQA](https://huggingface.co/datasets/openbookqa)        |

---

## 4️ Fact Verification / Truthfulness

Improves factual accuracy in answers.

| Dataset  | Purpose                            | HuggingFace Link                                     |
| -------- | ---------------------------------- | ---------------------------------------------------- |
| FEVER    | Claim verification from Wikipedia  | [FEVER](https://huggingface.co/datasets/fever)       |
| HoVer    | Multi-hop verification             | [Hover](https://huggingface.co/datasets/hover)       |
| VitaminC | Fact-checking for multiple domains | [VitaminC](https://huggingface.co/datasets/vitaminc) |


---

## 5️ Instruction-following / General QA

To improve accuracy across tasks, include instruction-following datasets.

| Dataset                 | Purpose                  | HuggingFace Link                                            |
| ----------------------- | ------------------------ | ----------------------------------------------------------- |
| Flan-T5 (small subsets) | Instruction-following    | [Flan-T5](https://huggingface.co/datasets/flan_t5)          |
| Alpaca (small)          | Instruction → response   | [Alpaca](https://huggingface.co/datasets/tatsu-lab/alpaca)  |
| ShareGPT                | Real human conversations | [ShareGPT](https://huggingface.co/datasets/openai/ShareGPT) |

---



## 6 Coding / Programming (Reasoning + GRPO)

### **A. Algorithmic reasoning (distillation-focused)**

| Dataset                 | Purpose                                       | HuggingFace Link                                                                                                                 |
| ----------------------- | --------------------------------------------- | -------------------------------------------------------------------------------------------------------------------------------- |
| CodeContests            | Competitive programming, reasoning + planning | [CodeContests](https://huggingface.co/datasets/huggingfaceH4/code_contests)                                                      |
| MBPP (full)             | Short Python algorithm tasks                  | [MBPP](https://huggingface.co/datasets/google-research-datasets/mbpp)                                                            |
| HumanEval / HumanEval-X | Classic Python coding evaluation              | [HumanEval](https://huggingface.co/datasets/openai/humaneval) / [HumanEval-X](https://huggingface.co/datasets/THUDM/humaneval-x) |

> **Approach:** Use question + task only, generate **COT reasoning + solution** via OSS120B. Perfect for step-by-step coding reasoning distillation.

---

### **B. Debugging / Repair (GRPO / RL-focused)**

| Dataset                     | Purpose                     | HuggingFace Link                                                               |
| --------------------------- | --------------------------- | ------------------------------------------------------------------------------ |
| DeepFix / Break-It / Fix-It | Real-world buggy code tasks | [SWE-bench Lite](https://huggingface.co/datasets/princeton-nlp/SWE-bench_Lite) |
| QuixBugs                    | Small debugging problems    | [QuixBugs](https://huggingface.co/datasets/quixbugs/quixbugs)                  |

> **Approach:** Feed buggy code + instructions to OSS120B → generate **step-by-step repair + corrected code**. Use for RL/GRPO fine-tuning.

---

### **C. Code Completion (breadth / optional)**

| Dataset                      | Purpose                                   | HuggingFace Link                                                                |
| ---------------------------- | ----------------------------------------- | ------------------------------------------------------------------------------- |
| TheStack v2 (curated subset) | Multi-language completion, short snippets | [TheStack v2](https://huggingface.co/datasets/bigcode/the-stack-v2-py)          |
| CodeParrot-clean             | Tiny Python snippets                      | [CodeParrot](https://huggingface.co/datasets/codeparrot/codeparrot-clean-valid) |


## 7 Creative Writing / Storytelling

| Dataset                 | Purpose                                                  | HuggingFace Link                                                  |
| ----------------------- | -------------------------------------------------------- | ----------------------------------------------------------------- |
| WritingPrompts          | Fictional story generation, character + plot development | [WritingPrompts](https://huggingface.co/datasets/writing_prompts) |
| FairyTaleQA             | Short stories + QA about story content                   | [FairyTaleQA](https://huggingface.co/datasets/fairytalesqa)       |
| StoryCloze / ROCStories | Short story sequences for plot reasoning                 | [ROCStories](https://huggingface.co/datasets/roc_stories)         |

> **Use for:** Teaching small models long-form creative reasoning and narrative consistency.

---

## 8 Creative Ideation / Brainstorming

| Dataset                        | Purpose                                          | HuggingFace Link                                                                    |
| ------------------------------ | ------------------------------------------------ | ----------------------------------------------------------------------------------- |
| CommonGen                      | Constrained sentence / idea generation           | [CommonGen](https://huggingface.co/datasets/commongen)                              |
| IdeaBench / ConceptNet stories | Novel idea generation / causal reasoning         | [IdeaBench](https://huggingface.co/datasets/ideabench) (small experimental subsets) |
| GoEmotions + prompt rewriting  | Creative prompts and brainstorming for responses | [GoEmotions](https://huggingface.co/datasets/go_emotions)                           |

> **Use for:** Step-by-step ideation, multiple candidate generation, or “thought expansion” tasks.

---

## 9 Summarization / Long-Form Reasoning

| Dataset                  | Purpose                                   | HuggingFace Link                                                       |
| ------------------------ | ----------------------------------------- | ---------------------------------------------------------------------- |
| CNN/DailyMail            | News article summarization (long → short) | [cnn_dailymail](https://huggingface.co/datasets/cnn_dailymail)         |
| XSum                     | Highly abstractive summarization          | [xsum](https://huggingface.co/datasets/xsum)                           |
| PubMed / arXiv Summaries | Scientific paper summarization            | [scientific_papers](https://huggingface.co/datasets/scientific_papers) |
| BIGPATENT                | Technical patent summarization            | [big_patent](https://huggingface.co/datasets/big_patent)               |

> **Use for:** Teaching the model long-context understanding, compression, and paraphrasing skills.

---

## 10 Basic Science / Reasoning

| Dataset                | Purpose                                      | HuggingFace Link                                                |
| ---------------------- | -------------------------------------------- | --------------------------------------------------------------- |
| CommonsenseQA          | Multi-choice commonsense + science reasoning | [CommonsenseQA](https://huggingface.co/datasets/commonsense_qa) |
| OpenBookQA             | Scientific facts + reasoning                 | [OpenBookQA](https://huggingface.co/datasets/openbookqa)        |
| ARC (Easy + Challenge) | Elementary to high school science QA         | [ARC](https://huggingface.co/datasets/ai2_arc)                  |

> **Use for:** General reasoning, basic science QA, and multi-step problem-solving.

---

## 11 Optional: Long-Form Multi-Step Reasoning / Multi-Hop QA

| Dataset     | Purpose                                         | HuggingFace Link                                           |
| ----------- | ----------------------------------------------- | ---------------------------------------------------------- |
| HotpotQA    | Multi-hop reasoning over Wikipedia passages     | [HotpotQA](https://huggingface.co/datasets/hotpot_qa)      |
| StrategyQA  | Implicit reasoning / multi-step                 | [StrategyQA](https://huggingface.co/datasets/strategy_qa)  |
| NarrativeQA | Story comprehension / reasoning about narrative | [NarrativeQA](https://huggingface.co/datasets/narrativeqa) |

## 12 Specialized Domains (Optional)

Optional if you want the model to be precise in professional domains:

| Domain                    | Dataset Examples   |
| ------------------------- | ------------------ |
| Legal                     | CaseQA, LegalBench |
| Science                   | PubMedQA, SciQ     |
| History / Wikipedia facts | HoPE, Wikidata QA  |


### 1. Domain : Math

1.1 GSM 8K

Dataset Summary:

    GSM8K (Grade School Math 8K) is a dataset of 8.5K high quality linguistically diverse grade school math word problems. The dataset was created to support the task of question answering on basic mathematical problems that require multi-step reasoning.

These problems take between 2 and 8 steps to solve.
Solutions primarily involve performing a sequence of elementary calculations using basic arithmetic operations (+ − ×÷) to reach the final answer.
A bright middle school student should be able to solve every problem: from the paper, "Problems require no concepts beyond the level of early Algebra, and the vast majority of problems can be solved without explicitly defining a variable."
Solutions are provided in natural language, as opposed to pure math expressions. From the paper: "We believe this is the most generally useful data format, and we expect it to shed light on the properties of large language models’ internal monologues""
Supported Tasks and Leaderboards
This dataset is generally used to test logic and math in language modelling. It has been used for many benchmarks, including the LLM Leaderboard.

In [3]:
##main dataset
dataset_master=pd.DataFrame()

##Load GSM8K
## Using main set of gsm8k
##socratis set is same questions but have different way of answering the question (with a step by step questioning approach)
## But either doesn't matter since we will be creating reasoning and response from oss120B
gsm8k = load_dataset('openai/gsm8k',  'main',   streaming=True)
gsm8k_train = pd.DataFrame(gsm8k['train'])
gsm8k_test = pd.DataFrame(gsm8k['test'])

gsm8k_train['split'] = 'train'
gsm8k_test['split'] = 'test'
dataset_master = pd.concat([gsm8k_train, gsm8k_test], ignore_index=True)
dataset_master = dataset_master.rename({'question':'input','answer':'source_answer'}, axis=1)
dataset_master['source'] = 'gsm8k'
dataset_master['domain'] = 'math'
dataset_master['ground_truth'] = dataset_master.apply(lambda row: extract_gsm8k_answer(row['source_answer']) if row['source'] == 'gsm8k' else None, axis=1)


In [ ]:

# open-r1/OpenR1-Math-220k
# Kinda huge dataset with 93K rows
# We will sample random 5K rows
# This data have more diverse and detailed math questions across different math category
# Geometry is also there and is an overshoot for text models but we will leave it

open_r1_math = load_dataset('open-r1/OpenR1-Math-220k',
                            streaming=True)['train'].select_columns(['problem', 'solution', 'answer', 'problem_type'
                                                                    ,'question_type','source'])
open_r1_math_df0=sample_hf_data(open_r1_math, sample_size=5000)
open_r1_math_df = convert_to_pandas(open_r1_math_df0, batch_size=1000)
open_r1_math_df = open_r1_math_df.rename(
    columns={
        'problem': 'input',
        'solution': 'source_answer',
        'answer': 'ground_truth'
    },
    inplace=False
)
open_r1_math_df['split'] = 'train'
open_r1_math_df['source'] = 'OpenR1-Math-220k'
open_r1_math_df['domain'] ='math'
dataset_master['problem_type'] = ''
dataset_master['question_type'] = ''
dataset_master1 = pd.concat([dataset_master.reset_index(drop=True), open_r1_math_df[dataset_master.columns].reset_index(drop=True)])

del open_r1_math_df, open_r1_math_df0, open_r1_math

Processing batch 0 

Processing batch 1 

Processing batch 2 

Processing batch 3 

Processing batch 4 



1.3 Math/AIME 2025

math-ai/aime25

In [ ]:
aime = load_dataset('math-ai/aime25',
                            streaming=True)['test'].select_columns(['problem','answer'])
aime_df = pd.DataFrame(aime)

aime_df.rename(columns={'problem':'input','answer':'ground_truth'}, inplace=True)
aime_df['source_answer']=aime_df['ground_truth']
aime_df['split'] = 'train'
aime_df['source'] = 'Math/AIME 2025'
aime_df['domain'] ='math'
aime_df['problem_type']=''
aime_df['question_type']=''
dataset_master2 = pd.concat([dataset_master1.reset_index(drop=True), aime_df[dataset_master1.columns].reset_index(drop=True)])

del aime_df, aime,dataset_master

1.4 ChilleD/SVAMP
Small multi-step arithmetic problems

In [ ]:
svamp_train = load_dataset('ChilleD/SVAMP',
                            streaming=True)['train'].select_columns(['question_concat','Answer','Type'])
svamp_test = load_dataset('ChilleD/SVAMP',
                            streaming=True)['test'].select_columns(['question_concat','Answer','Type'])

svamp_df_train = pd.DataFrame(svamp_train)
svamp_df_train['split'] = 'train'
svamp_df_test = pd.DataFrame(svamp_test)
svamp_df_test['split'] = 'test'
svamp_df = pd.concat([svamp_df_train, svamp_df_test], ignore_index=True)


svamp_df.rename(columns={'question_concat':'input','Answer':'ground_truth'}, inplace=True)
svamp_df['source_answer']=svamp_df['ground_truth']

svamp_df['source'] = 'ChilleD/SVAMP'
svamp_df['domain'] ='math'
svamp_df['problem_type']=''
svamp_df['question_type']=''

dataset_master3 = pd.concat([dataset_master2.reset_index(drop=True), svamp_df[dataset_master2.columns].reset_index(drop=True)])
del svamp_df, svamp_df_train, svamp_df_test, dataset_master1

In [29]:
dataset_master3.groupby(['source','split']).size()

source            split
ChilleD/SVAMP     test      300
                  train     700
Math/AIME 2025    train      30
OpenR1-Math-220k  train    5000
gsm8k             test     1319
                  train    7473
dtype: int64

In [33]:
##Add a uid to dataset with prefix math
dataset_master3['uid'] = ['math' + str(i) for i in range(1, len(dataset_master3) + 1)]

In [35]:
##Save the math base dataset
dataset_master3.to_parquet('../data/raw-data/math_base_dataset.parquet')
dataset_master3.to_csv('../data/raw-data/math_base_dataset.csv', index=False)

### 2. Domain : Code

2.1 MBPP

In [4]:
code_master0 = pd.DataFrame()

In [12]:
mbpp_train = convert_to_pandas(load_dataset('google-research-datasets/mbpp',
                            streaming=True)['train'].select_columns(['text', 'code', 'test_list']), batch_size=1000)

mbpp_test = convert_to_pandas(load_dataset('google-research-datasets/mbpp',
                            streaming=True)['test'].select_columns(['text', 'code', 'test_list']), batch_size=1000)
mbpp_validation = convert_to_pandas(load_dataset('google-research-datasets/mbpp',
                            streaming=True)['validation'].select_columns(['text', 'code', 'test_list']), batch_size=1000)
mbpp_prompt = convert_to_pandas(load_dataset('google-research-datasets/mbpp',
                            streaming=True)['prompt'].select_columns(['text', 'code', 'test_list']), batch_size=1000)

mbpp_train['split'] = 'train'
mbpp_test['split'] = 'test'
mbpp_validation['split'] = 'validation'
mbpp_prompt['split'] = 'prompt'

code_master0 = pd.concat([mbpp_train, mbpp_test, mbpp_validation, mbpp_prompt], ignore_index=True)
code_master0.rename(columns={'text':'input','code':'source_answer','test_list':'ground_truth'}, inplace=True)
code_master0['source'] = 'google-research-datasets/mbpp'
code_master0['domain'] = 'code'
code_master0['problem_type']='Algorithmic reasoning'
code_master0['question_type']='Short Python algorithm tasks'

del mbpp_train, mbpp_test, mbpp_validation, mbpp_prompt, mbpp

Processing batch 0 

Processing batch 0 

Processing batch 0 

Processing batch 0 



2.2 nvidia/OpenCodeReasoning

In [99]:
nvidia_code = load_dataset('nvidia/OpenCodeReasoning', 'split_0',streaming=True)
# Filter by difficulty
filtered = nvidia_code.filter(lambda x: x['difficulty'] in ['EASY', 'MEDIUM','UNKNOWN_DIFFICULTY']).select_columns(['input','solution','source','dataset','difficulty','split'])
print(f"Total easy/medium samples: {len(filtered)}")
# Shuffle and sample
sampled = sample_hf_data(nvidia_code['split_0'], sample_size=10000)
# Convert to pandas
nvidia_code_df = convert_to_pandas(sampled, batch_size=1000)

Total easy/medium samples: 1
Processing batch 0 

Processing batch 1 

Processing batch 2 

Processing batch 3 

Processing batch 4 

Processing batch 5 

Processing batch 6 

Processing batch 7 

Processing batch 8 

Processing batch 9 



In [ ]:
nvidia_code_df.rename(columns={'source':'question_type','solution':'source_answer'
                               ,'dataset':'problem_type'}, inplace=True)
nvidia_code_df['split'] = 'train'
nvidia_code_df['ground_truth'] = ''
nvidia_code_df['source'] = 'nvidia/OpenCodeReasoning'
nvidia_code_df['domain'] ='code'
code_master0['difficulty'] = 'UNKNOWN_DIFFICULTY'

code_master1 = pd.concat([code_master0.reset_index(drop=True), nvidia_code_df[code_master0.columns].reset_index(drop=True)], ignore_index=True)
del nvidia_code_df, nvidia_code, sampled, filtered, code_master0

2.3 codeparrot/apps

In [129]:
codeparrot_apps

,id,input,source_answer,ground_truth,difficulty,url,starter_code,split,source,domain,problem_type,question_type
0,0,Polycarp has $n$ different binary words. A wor...,"[""for _ in range(int(input())):\n n = int(i...","{\n ""inputs"": [\n ""4\n4\n0001\n1000\n0011\...",UNKNOWN_DIFFICULTY,https://codeforces.com/problemset/problem/1259/D,,train,codeparrot/apps,code,Coding,Code generation
1,1,Mikhail walks on a Cartesian plane. He starts ...,"[""q=int(input())\n\nfor e in range(q):\n x,...","{\n ""inputs"": [\n ""3\n2 2 3\n4 3 7\n10 1 9...",UNKNOWN_DIFFICULTY,https://codeforces.com/problemset/problem/1036/B,,train,codeparrot/apps,code,Coding,Code generation
2,2,"You are given three sequences: $a_1, a_2, \ldo...","[""import sys\nimport random\nfrom fractions im...","{\n ""inputs"": [\n ""5\n3\n1 1 1\n2 2 2\n3 3...",UNKNOWN_DIFFICULTY,https://codeforces.com/problemset/problem/1408/A,,train,codeparrot/apps,code,Coding,Code generation
3,3,"You have $n$ barrels lined up in a row, number...","[""def solve():\n n, k = map(int,input().spl...","{\n ""inputs"": [\n ""2\n4 1\n5 5 5 5\n3 2\n0...",UNKNOWN_DIFFICULTY,https://codeforces.com/problemset/problem/1430/B,,train,codeparrot/apps,code,Coding,Code generation
4,4,"You are given a permutation $p=[p_1, p_2, \ldo...","[""for _ in range(int(input())):\n input()\n...","{\n ""inputs"": [\n ""3\n6\n4 5 1 3 2 6\n5\n5...",UNKNOWN_DIFFICULTY,https://codeforces.com/problemset/problem/1265/B,,train,codeparrot/apps,code,Coding,Code generation
...,...,...,...,...,...,...,...,...,...,...,...,...
4995,4995,Another rewarding day in the fast-paced world ...,"[""class HTMLGen:\n def __init__(self):\n ...","{""fn_name"": ""__init__"", ""inputs"": [], ""outputs...",UNKNOWN_DIFFICULTY,https://www.codewars.com/kata/54eecc187f9142cc...,\ndef __init__(self):\n\t,train,codeparrot/apps,code,Coding,Code generation
4996,4996,## **Instructions**\n\nThe goal of this kata i...,"[""def fibs_fizz_buzz(n):\n a, b, out = 0, 1...","{""fn_name"": ""fibs_fizz_buzz"", ""inputs"": [], ""o...",UNKNOWN_DIFFICULTY,https://www.codewars.com/kata/57bf599f102a39bb...,\ndef fibs_fizz_buzz(n):\n\t,train,codeparrot/apps,code,Coding,Code generation
4997,4997,"The function sigma 1, σ1 in mathematics, is kn...","[""cache = {}\ndef sum_div(x):\n if x not in...","{""fn_name"": ""sigma1"", ""inputs"": [], ""outputs"":...",UNKNOWN_DIFFICULTY,https://www.codewars.com/kata/5619dbc22e69620e...,\ndef sigma1(n):\n\t,train,codeparrot/apps,code,Coding,Code generation
4998,4998,The principal of a school likes to put challen...,"[""def wanted_words(vowels, consonants, forbidd...","{""fn_name"": ""wanted_words"", ""inputs"": [[1, 7, ...",UNKNOWN_DIFFICULTY,https://www.codewars.com/kata/580be55ca671827c...,"\ndef wanted_words(n, m, forbid_let):\n\t",train,codeparrot/apps,code,Coding,Code generation


In [ ]:
# 2.3 codeparrot/apps using JSONL
import json
from urllib.parse import urlparse

url = 'https://codeforces.com/problemset/problem/1259/D'
domain = urlparse(url).netloc.split('.')[0]
print(domain)  # Output: codeforces

codeparrot_apps0= load_dataset(
    'json',
    data_files='https://huggingface.co/datasets/codeparrot/apps/resolve/main/train.jsonl',
    split='train',
    streaming=True
)

codeparrot_apps= convert_to_pandas(codeparrot_apps0, batch_size=1000)

# # Standardize column names
codeparrot_apps.rename(columns={'question': 'input', 'solutions': 'source_answer',
                                'input_output':'ground_truth'}, inplace=True)
codeparrot_apps['split'] = 'train'
codeparrot_apps['source'] = 'codeparrot/apps'
codeparrot_apps['domain'] = 'code'
codeparrot_apps['problem_type'] = 'Coding'
codeparrot_apps['question_type'] = codeparrot_apps['url'].apply(
    lambda x: re.search(r'https?://(?:www\.)?([^/.]+)', x).group(1) if x and re.search(r'https?://(?:www\.)?([^/.]+)', x) else 'unknown'
)



codeparrot_apps['difficulty'] = 'UNKNOWN_DIFFICULTY'
code_master2 = pd.concat([code_master1.reset_index(drop=True), codeparrot_apps[code_master1.columns].reset_index(drop=True)], ignore_index=True)


codeforces
Processing batch 0 

Processing batch 1 

Processing batch 2 

Processing batch 3 

Processing batch 4 



2.3 CodeAlpaca -20

In [158]:
code_alpha = load_dataset('sahil2801/CodeAlpaca-20k',
    streaming=True)['train']

code_alpaca_df = convert_to_pandas(code_alpha, batch_size=1000)
code_alpaca_df['input_prompt'] = code_alpaca_df['instruction'] + '\n' + code_alpaca_df['input']
code_alpaca_df['base_input'] = code_alpaca_df['input']
code_alpaca_df.drop(columns=['instruction','input'], inplace=True)
code_alpaca_df.rename(columns={'input_prompt': 'input', 'output': 'source_answer'}, inplace=True)
code_alpaca_df['split'] = 'train'
code_alpaca_df['ground_truth'] = ''
code_alpaca_df['source'] = 'sahil2801/CodeAlpaca-20k'
code_alpaca_df['domain'] = 'code'
code_alpaca_df['problem_type'] = 'code generation'
code_alpaca_df['question_type'] = 'code generation by instruction'
code_alpaca_df['difficulty'] = 'UNKNOWN_DIFFICULTY'

Processing batch 0 

Processing batch 1 

Processing batch 2 

Processing batch 3 

Processing batch 4 

Processing batch 5 

Processing batch 6 

Processing batch 7 

Processing batch 8 

Processing batch 9 

Processing batch 10 

Processing batch 11 

Processing batch 12 

Processing batch 13 

Processing batch 14 

Processing batch 15 

Processing batch 16 

Processing batch 17 

Processing batch 18 

Processing batch 19 

Processing batch 20 



In [159]:
code_alpaca_df

,source_answer,input,base_input,split,ground_truth,source,domain,problem_type,question_type,difficulty
0,"arr = [2, 4, 6, 8, 10]",Create an array of length 5 which contains all...,,train,,sahil2801/CodeAlpaca-20k,code,code generation,code generation by instruction,UNKNOWN_DIFFICULTY
1,Height of triangle = opposite side length * si...,Formulate an equation to calculate the height ...,,train,,sahil2801/CodeAlpaca-20k,code,code generation,code generation by instruction,UNKNOWN_DIFFICULTY
2,"def replace(self, replace_with):\n new_stri...",Write a replace method for a string class whic...,"string = ""Hello World!""\nreplace_with = ""Greet...",train,,sahil2801/CodeAlpaca-20k,code,code generation,code generation by instruction,UNKNOWN_DIFFICULTY
3,"arr = [3, 6, 9, 12, 15, 18, 21, 24, 27, 30, 33...",Create an array of length 15 containing number...,,train,,sahil2801/CodeAlpaca-20k,code,code generation,code generation by instruction,UNKNOWN_DIFFICULTY
4,def find_num_distinct_states(matrix):\n sta...,Write a function to find the number of distinc...,"matrix = [[1, 0, 0],\n [1, 0, 1],\n ...",train,,sahil2801/CodeAlpaca-20k,code,code generation,code generation by instruction,UNKNOWN_DIFFICULTY
...,...,...,...,...,...,...,...,...,...,...
20017,SELECT DISTINCT data_type \nFROM products \nWH...,Identify and display all the data types within...,,train,,sahil2801/CodeAlpaca-20k,code,code generation,code generation by instruction,UNKNOWN_DIFFICULTY
20018,"num1 = int(input(""Enter the first number: ""))\...",Create a basic Python script to accept two num...,,train,,sahil2801/CodeAlpaca-20k,code,code generation,code generation by instruction,UNKNOWN_DIFFICULTY
20019,using System;\n\npublic class Program \n{\n ...,Create a basic C# program to print out the cur...,,train,,sahil2801/CodeAlpaca-20k,code,code generation,code generation by instruction,UNKNOWN_DIFFICULTY
20020,SELECT e.name \nFROM employees e \nWHERE e.sal...,Find and display all the employees who earn mo...,,train,,sahil2801/CodeAlpaca-20k,code,code generation,code generation by instruction,UNKNOWN_DIFFICULTY


In [160]:
code_master3= pd.concat([code_master2.reset_index(drop=True), code_alpaca_df[code_master2.columns].reset_index(drop=True)], ignore_index=True)

4. iamtarun/python_code_instructions_18k_alpaca


- specific for python
- to improvfe python coding

In [165]:


python_code_alpaca = load_dataset('iamtarun/python_code_instructions_18k_alpaca',
    streaming=True)['train'].select_columns(['prompt','output'])

python_code_alpaca_df = convert_to_pandas(python_code_alpaca, batch_size=1000)
python_code_alpaca_df.rename(columns={'prompt': 'input', 'output': 'source_answer'}, inplace=True)
python_code_alpaca_df['split'] = 'train'
python_code_alpaca_df['ground_truth'] = ''
python_code_alpaca_df['source'] = 'iamtarun/python_code_instructions_18k_alpaca'
python_code_alpaca_df['domain'] = 'code'
python_code_alpaca_df['problem_type'] = 'python code generation'
python_code_alpaca_df['question_type'] = 'python code generation by instruction'
python_code_alpaca_df['difficulty'] = 'UNKNOWN_DIFFICULTY'

Processing batch 0 

Processing batch 1 

Processing batch 2 

Processing batch 3 

Processing batch 4 

Processing batch 5 

Processing batch 6 

Processing batch 7 

Processing batch 8 

Processing batch 9 

Processing batch 10 

Processing batch 11 

Processing batch 12 

Processing batch 13 

Processing batch 14 

Processing batch 15 

Processing batch 16 

Processing batch 17 

Processing batch 18 



In [166]:
code_master4= pd.concat([code_master3.reset_index(drop=True), python_code_alpaca_df[code_master3.columns].reset_index(drop=True)], ignore_index=True)

In [170]:
print(code_master4.groupby(['source','question_type','difficulty']).size())

source                                        question_type                          difficulty        
codeparrot/apps                               atcoder                                UNKNOWN_DIFFICULTY       61
                                              codechef                               UNKNOWN_DIFFICULTY     1112
                                              codeforces                             UNKNOWN_DIFFICULTY      477
                                              codewars                               UNKNOWN_DIFFICULTY     2515
                                              hackerrank                             UNKNOWN_DIFFICULTY       96
                                              leetcode                               UNKNOWN_DIFFICULTY      739
google-research-datasets/mbpp                 Short Python algorithm tasks           UNKNOWN_DIFFICULTY      974
iamtarun/python_code_instructions_18k_alpaca  python code generation by instruction  UNKNOWN_DIFFICULTY  

greengerong/leetcode Use if we need leetcode to enhance different programming languages individually later

In [173]:
# Convert any list columns to JSON strings so pyarrow can write parquet
for col in code_master4.columns:
    if code_master4[col].apply(lambda x: isinstance(x, list)).any():
        code_master4[col] = code_master4[col].apply(lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, list) else x)


In [176]:
##Add a uid to dataset with prefix code
code_master4['uid'] = ['code' + str(i) for i in range(1, len(code_master4) + 1)]
code_master4.to_csv('../data/raw-data/code_base_dataset.csv', index=False)
code_master4.to_parquet('../data/raw-data/code_base_dataset.parquet', index=False)